## 22. عرضه و روند قیمت

تحلیل زمانی باید به تفکیک مناسب انجام شود:

- ماه
- شهر
- محله، در صورت کفایت نمونه
- نوع ملک
- رژیم قیمت

شاخص‌های پیشنهادی:

- تعداد آگهی خام
- تعداد آگهی پس از Deduplication
- میانه قیمت هر مترمربع
- تغییر ماه‌به‌ماه
- IQR قیمت
- تعداد مناطق دارای داده کافی

### محدودیت

تغییر تعداد آگهی می‌تواند ناشی از تغییر رفتار کاربران، پوشش پلتفرم، Duplicate یا فصل باشد.
آن را مستقیماً برابر با تغییر موجودی واقعی بازار در نظر نگیرید.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_feather("../Outputs/21_df.feather")

In [4]:
# -----------------------------
# 1. آماده سازی
# -----------------------------

# df['created_at'] = pd.to_datetime(
#     df['created_at_month'],
#     errors='coerce'
# )

df['sale_price_per_sqm'] = pd.to_numeric(
    df['sale_price_per_sqm'],
    errors='coerce'
)

df['month'] = df['created_at_month'].dt.to_period('M').dt.to_timestamp()


# -----------------------------
# 2. ستون های گروه بندی
# -----------------------------

group_cols = [
    'month',
    'city_slug',
    'neighborhood_slug',
    'cat2_slug'
]


# -----------------------------
# 3. تعداد آگهی
# -----------------------------

ads = df.groupby(
    group_cols,
    observed=True
).size().reset_index(
    name='ads_count'
)


# -----------------------------
# 4. قیمت معتبر
# -----------------------------

price_df = df[
    df['sale_price_per_sqm'].notna() &
    (df['sale_price_per_sqm'] > 0)
].copy()


# -----------------------------
# 5. آمار قیمت
# -----------------------------

price_stats = price_df.groupby(
    group_cols,
    observed=True
)['sale_price_per_sqm'].agg(
    median_price_per_sqm='median',
    q1_price_per_sqm=lambda x: x.quantile(0.25),
    q3_price_per_sqm=lambda x: x.quantile(0.75),
    price_count='count'
).reset_index()


# -----------------------------
# 6. IQR
# -----------------------------

price_stats['price_iqr'] = (
    price_stats['q3_price_per_sqm']
    -
    price_stats['q1_price_per_sqm']
)


# -----------------------------
# 7. ترکیب
# -----------------------------

supply_trend = ads.merge(
    price_stats,
    on=group_cols,
    how='left'
)


# -----------------------------
# 8. مرتب سازی
# -----------------------------

supply_trend = supply_trend.sort_values(
    [
        'city_slug',
        'neighborhood_slug',
        'cat2_slug',
        'month'
    ]
)


# -----------------------------
# 9. تغییر ماهانه قیمت
# -----------------------------

trend_cols = [
    'city_slug',
    'neighborhood_slug',
    'cat2_slug'
]

supply_trend['price_mom_pct'] = (
    supply_trend
    .groupby(
        trend_cols,
        observed=True
    )['median_price_per_sqm']
    .pct_change()
    * 100
)


# -----------------------------
# 10. تغییر ماهانه تعداد آگهی
# -----------------------------

supply_trend['ads_mom_pct'] = (
    supply_trend
    .groupby(
        trend_cols,
        observed=True
    )['ads_count']
    .pct_change()
    * 100
)


# -----------------------------
# 11. کفایت نمونه
# -----------------------------

MIN_SAMPLE = 20

supply_trend['enough_data'] = (
    supply_trend['price_count'] >= MIN_SAMPLE
)


print(supply_trend.head())

C:\Users\lenovo\AppData\Local\Temp\ipykernel_9340\3108103982.py:119: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


           month city_slug neighborhood_slug        cat2_slug  ads_count  \
2326  2024-05-01     ahvaz    amaniyeh-ahvaz  commercial-rent          2   
6424  2024-06-01     ahvaz    amaniyeh-ahvaz  commercial-rent          3   
10726 2024-07-01     ahvaz    amaniyeh-ahvaz  commercial-rent          2   
15073 2024-08-01     ahvaz    amaniyeh-ahvaz  commercial-rent          3   
19341 2024-09-01     ahvaz    amaniyeh-ahvaz  commercial-rent          2   

       median_price_per_sqm  q1_price_per_sqm  q3_price_per_sqm  price_count  \
2326                    NaN               NaN               NaN          NaN   
6424                    NaN               NaN               NaN          NaN   
10726                   NaN               NaN               NaN          NaN   
15073                   NaN               NaN               NaN          NaN   
19341                   NaN               NaN               NaN          NaN   

       price_iqr  price_mom_pct  ads_mom_pct  enough_data  
23

In [5]:
supply_trend.to_csv(
    '../Outputs/supply_trend.csv',
    index=False,
    encoding='utf-8-sig'
)